# 维度二：风险归因分解

基于真实 provided 风险归因死亡数据展示最新年份的全球与地区风险贡献结构。


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    if ROOT.name == "notebooks" and (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    elif (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise RuntimeError("无法定位仓库根目录，请从项目根目录或 notebooks/ 目录启动 Notebook。")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)

def read_api(relpath: str):
    payload = read_json(ROOT / relpath)
    return payload.get("data", payload)

def show_image(relpath: str, width: int = 900):
    display(Image(filename=str(ROOT / relpath), width=width))

def require(*relpaths: str):
    missing = [path for path in relpaths if not (ROOT / path).exists()]
    if missing:
        raise FileNotFoundError(
            "缺少以下产物，请先运行 `PYTHONPATH=src python -m hdi.data.integrator` 和 "
            "`PYTHONPATH=src python -m hdi.analysis.competition`:\n- " + "\n- ".join(missing)
        )


In [ ]:
require(
    "api_output/dim2/paf.json",
    "api_output/dim2/shapley_global.json",
    "reports/figures/fig04_dim2_global_risk_bar.png",
    "reports/figures/fig05_dim2_region_heatmap.png",
)

paf = pd.DataFrame(read_api("api_output/dim2/paf.json"))
shapley = pd.DataFrame(read_api("api_output/dim2/shapley_global.json"))

latest_year = paf["year"].max()
latest = paf[paf["year"] == latest_year].copy()
top_global = latest.groupby("risk_factor", as_index=False)["attributable_deaths"].sum().sort_values("attributable_deaths", ascending=False).head(10)

display(top_global)
display(shapley)


In [ ]:
show_image("reports/figures/fig04_dim2_global_risk_bar.png")


In [ ]:
show_image("reports/figures/fig05_dim2_region_heatmap.png")
